# Notebook 4: The Oklahoma Experiment

Oklahoma's induced seismicity crisis (2009–2025) constitutes one of the clearest natural experiments in crustal mechanics: a known perturbation (massive wastewater injection), a measurable response (orders-of-magnitude seismicity increase), and a partial reversal (regulatory volume reductions). This notebook tracks the full statistical lifecycle of that experiment using earthquake catalogs from USGS ComCat and wastewater injection volumes from the Oklahoma Corporation Commission (OCC) Underground Injection Control (UIC) program.

We define five phases informed by the published literature, quantify rate changes with chi-squared significance tests, measure spatial migration via convex hull areas and permutation-tested centroid shifts, compare b-values at both per-phase and common Mc, and assess whether post-regulation seismicity has returned to pre-injection levels.

In [1]:
import matplotlib
matplotlib.use('Agg')

import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist

sys.path.insert(0, ".")
from src.catalog import estimate_mc
from src.gutenberg_richter import estimate_b_value, bootstrap_b_value, rolling_b_value
from src.interevent import compute_interevent_times, fit_distributions, classify_regime
from src.external_data import load_occ_uic_volumes, aggregate_oklahoma_injection
from src.plotting import setup_style, save_figure, plot_oklahoma_timeline

setup_style()

In [2]:
df = pd.read_csv("data/earthquake_catalog_oklahoma.csv")
df["time"] = pd.to_datetime(df["time"], format="ISO8601", utc=True)
print(f"Loaded {len(df):,} Oklahoma events")
print(f"Date range: {df['time'].min()} → {df['time'].max()}")
print(f"Magnitude range: {df['mag'].min():.1f} → {df['mag'].max():.1f}")

Loaded 31,187 Oklahoma events
Date range: 2000-10-08 10:16:23.780000+00:00 → 2025-12-31 13:16:31.969000+00:00
Magnitude range: 1.0 → 5.8


## 4.1a Injection Volume Data

We load monthly wastewater injection volumes from the Oklahoma Corporation Commission (OCC) UIC well data (2006-2024). This is the key independent variable linking industrial activity to seismicity.

**Important caveat on temporal resolution:** OCC data before 2011 reports annual totals only, which we assign to mid-year (month 6). This means pre-2011 "monthly" values are artifacts of this assignment. For the lag-correlation analysis below, we restrict to 2011+ data where true monthly reporting exists, avoiding spurious correlation patterns from the annual-to-monthly conversion.

In [3]:
# Load OCC UIC injection volumes (2006-2024, monthly per-well)
uic_df = load_occ_uic_volumes(years=list(range(2006, 2025)))
print(f"Total UIC records: {len(uic_df):,}")

# Aggregate to monthly Oklahoma totals
injection_monthly = aggregate_oklahoma_injection(uic_df)
print(f"\nMonthly injection volume summary ({injection_monthly['date'].min().year}-{injection_monthly['date'].max().year}):")
print(f"  Peak month: {injection_monthly.loc[injection_monthly['total_volume_bbl'].idxmax(), 'date'].strftime('%Y-%m')} "
      f"({injection_monthly['total_volume_bbl'].max()/1e6:.0f}M bbl)")
print(f"  Last month: {injection_monthly.iloc[-1]['date'].strftime('%Y-%m')} "
      f"({injection_monthly.iloc[-1]['total_volume_bbl']/1e6:.0f}M bbl)")
print(f"  Peak active wells: {injection_monthly['n_wells'].max():,}")

Total UIC records: 1,935,853



Monthly injection volume summary (2006-2024):
  Peak month: 2010-06 (1800M bbl)
  Last month: 2024-12 (106M bbl)
  Peak active wells: 8,253


## 4.1 Phase Definition

We divide Oklahoma's seismicity into five phases based on literature-informed boundaries (e.g., Ellsworth 2013, Langenbruch & Zoback 2016, Langenbruch et al. 2018). The boundaries align with known inflection points in injection practice and regulatory action:

- **Baseline (2000-2008):** Pre-injection-crisis background seismicity.
- **Onset (2009-2012):** Initial seismicity increase as disposal volumes ramp up.
- **Surge (2013-mid 2016):** Peak induced seismicity, including M5+ events.
- **Regulation (mid 2016-2019):** Mandatory volume reductions take effect.
- **Recovery (2020-2025):** Post-regulation period, testing whether seismicity returns to background.

These boundaries could alternatively be derived via data-driven changepoint detection (as in Notebook 01), but the literature-informed definitions allow direct comparison with prior studies.

In [4]:
phases = {
    "Baseline": ("2000-01-01", "2009-01-01"),
    "Onset": ("2009-01-01", "2013-01-01"),
    "Surge": ("2013-01-01", "2016-07-01"),
    "Regulation": ("2016-07-01", "2020-01-01"),
    "Recovery": ("2020-01-01", "2026-01-01"),
}

phase_colors = {
    "Baseline": "#457B9D",
    "Onset": "#2A9D8F",
    "Surge": "#E63946",
    "Regulation": "#F4A261",
    "Recovery": "#6A4C93",
}

# Create phase boundaries as tz-aware Timestamps
phase_starts = {name: pd.Timestamp(start, tz="UTC") for name, (start, end) in phases.items()}
phase_ends = {name: pd.Timestamp(end, tz="UTC") for name, (start, end) in phases.items()}

def assign_phase(t):
    for name in phases:
        if phase_starts[name] <= t < phase_ends[name]:
            return name
    return "Unknown"

df["phase"] = df["time"].apply(assign_phase)
print(df["phase"].value_counts())

phase
Recovery      14767
Surge          9923
Regulation     5965
Onset           488
Baseline         44
Name: count, dtype: int64


## 4.2 Seismicity Timeline

Monthly event counts (M2.5+) and rolling b-value across the full 25-year record, with phase boundaries overlaid. The timeline provides the first visual overview of the induced seismicity lifecycle.

In [5]:
# Monthly event counts (M2.5+)
mc_global = estimate_mc(df["mag"].values)
df_above = df[df["mag"] >= max(mc_global, 2.5)].copy()
df_above["month"] = df_above["time"].dt.to_period("M").dt.to_timestamp()
monthly = df_above.groupby("month").size().reset_index(name="count")

# Rolling b-value
rolling = rolling_b_value(df_above["time"].values, df_above["mag"].values,
                          mc=max(mc_global, 2.5), window_size=100, step=20)
rolling["center_time"] = pd.to_datetime(rolling["center_time"])

# Phase boundaries
phase_boundaries = [(pd.Timestamp(start, tz="UTC"), name) 
                    for name, (start, _) in phases.items()
                    if name != "Baseline"]

fig, ax = plt.subplots(figsize=(14, 7))
plot_oklahoma_timeline(monthly, rolling, phase_boundaries, ax=ax)
save_figure(fig, "04_oklahoma_timeline")
plt.show()

/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_5987/1091490013.py:4: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df_above["month"] = df_above["time"].dt.to_period("M").dt.to_timestamp()


/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_5987/1091490013.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4.2a Seismicity vs. Injection Volume

This is the central figure: monthly earthquake counts overlaid with total monthly wastewater injection volumes from OCC records. The dual-axis plot reveals the temporal relationship between injection activity and seismicity.

We quantify this relationship using Spearman rank correlation rather than Pearson, because the injection-seismicity relationship is expected to be monotonic but not necessarily linear, and Spearman is robust to outliers and non-normal distributions. The correlation analysis uses only 2011+ monthly data to avoid artifacts from the annual-to-monthly assignment in pre-2011 OCC records. We test lags from 0 to 12 months to capture pressure diffusion delays.

In [6]:
fig, ax1 = plt.subplots(figsize=(14, 7))

# Seismicity (left axis)
ax1.bar(monthly["month"], monthly["count"], width=25, color="#E63946", alpha=0.6, label="Earthquakes (M2.5+)")
ax1.set_xlabel("Date")
ax1.set_ylabel("Monthly Earthquake Count", color="#E63946")
ax1.tick_params(axis="y", labelcolor="#E63946")

# Injection volume (right axis)
ax2 = ax1.twinx()
ax2.plot(injection_monthly["date"], injection_monthly["total_volume_bbl"] / 1e6,
         color="#457B9D", linewidth=2, label="Injection Volume")
ax2.fill_between(injection_monthly["date"], 0, injection_monthly["total_volume_bbl"] / 1e6,
                 alpha=0.15, color="#457B9D")
ax2.set_ylabel("Monthly Injection Volume (million bbl)", color="#457B9D")
ax2.tick_params(axis="y", labelcolor="#457B9D")

# Phase boundaries
for name, (start, _) in phases.items():
    if name != "Baseline":
        ax1.axvline(pd.Timestamp(start), color="black", linestyle=":", alpha=0.4)
        ax1.text(pd.Timestamp(start), ax1.get_ylim()[1] * 0.95, f" {name}", fontsize=9, ha="left")

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

ax1.set_title("Oklahoma Seismicity vs. Wastewater Injection Volume", fontsize=14, fontweight="bold")
fig.tight_layout()
save_figure(fig, "04_injection_vs_seismicity")
plt.show()

# Compute lag correlation using Spearman rank correlation.
# We use Spearman because the injection-seismicity relationship is likely
# monotonic but not necessarily linear, and Spearman is robust to outliers.
# We filter to 2011 onward because 2006-2010 OCC data uses annual totals
# assigned to month 6, which creates spurious artifacts in monthly correlations.
from scipy.stats import spearmanr

monthly_merged = monthly.copy()
monthly_merged["month_dt"] = pd.to_datetime(monthly_merged["month"])
inj = injection_monthly[["date", "total_volume_bbl"]].copy()
inj["month_dt"] = inj["date"].dt.to_period("M").dt.to_timestamp()

merged = monthly_merged.merge(inj, on="month_dt", how="inner")
# Filter to 2011+ where true monthly injection data exists
merged = merged[merged["month_dt"] >= "2011-01-01"].reset_index(drop=True)

if len(merged) > 10:
    # Test lags 0 to 12 months
    print("Spearman rank correlations (injection vs. seismicity, 2011 onward):")
    best_rho, best_lag, best_p = 0, 0, 1.0
    for lag in range(0, 13):
        if lag == 0:
            rho, p = spearmanr(merged["total_volume_bbl"], merged["count"])
        else:
            rho, p = spearmanr(merged["total_volume_bbl"].iloc[:-lag], merged["count"].iloc[lag:])
        if abs(rho) > abs(best_rho):
            best_rho, best_lag, best_p = rho, lag, p
        if lag in [0, 3, 6, 9, 12]:
            print(f"  Lag {lag:2d} months: rho={rho:.3f}, p={p:.2e}")

    print(f"\nPeak correlation: rho={best_rho:.3f} at lag={best_lag} months (p={best_p:.2e})")
    rho0, p0 = spearmanr(merged["total_volume_bbl"], merged["count"])
    print(f"Zero-lag correlation: rho={rho0:.3f}, p={p0:.2e}")

    # Interpretation
    if best_p < 0.01:
        print(f"\nInterpretation: Statistically significant monotonic association (p<0.01) "
              f"between injection volume and seismicity at {best_lag}-month lag.")
    elif best_p < 0.05:
        print(f"\nInterpretation: Marginally significant association (p<0.05) "
              f"at {best_lag}-month lag. Relationship may be confounded by regulatory timing.")
    else:
        print(f"\nInterpretation: No statistically significant monotonic correlation detected. "
              f"This may reflect nonlinear pressure diffusion dynamics or threshold effects "
              f"that simple lag-correlation cannot capture.")

Spearman rank correlations (injection vs. seismicity, 2011 onward):
  Lag  0 months: rho=0.696, p=2.52e-25
  Lag  3 months: rho=0.759, p=8.20e-32
  Lag  6 months: rho=0.771, p=8.83e-33
  Lag  9 months: rho=0.785, p=4.93e-34
  Lag 12 months: rho=0.819, p=1.45e-38

Peak correlation: rho=0.819 at lag=12 months (p=1.45e-38)
Zero-lag correlation: rho=0.696, p=2.52e-25

Interpretation: Statistically significant monotonic association (p<0.01) between injection volume and seismicity at 12-month lag.


/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_5987/3061955624.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4.3 Spatial Evolution by Phase

Mapping all events per phase, colored by depth, reveals how the spatial footprint of seismicity expanded from a few localized clusters (Onset) to a broad swath of north-central Oklahoma (Surge), then partially contracted.

We quantify spatial extent using the convex hull area (in deg²) of event locations per phase, and test whether the centroid shift between Surge and Recovery is statistically significant using a permutation test (1000 iterations). The centroid migration metric captures the net spatial displacement of seismicity as injection patterns changed.

In [7]:
fig, axes = plt.subplots(1, 5, figsize=(20, 5), sharey=True)

for i, (phase_name, (start, end)) in enumerate(phases.items()):
    ax = axes[i]
    mask = df["phase"] == phase_name
    phase_df = df[mask]
    
    sc = ax.scatter(phase_df["longitude"], phase_df["latitude"],
                    c=phase_df["depth"], cmap="viridis_r", s=2, alpha=0.3,
                    vmin=0, vmax=15)
    ax.set_xlim(-100, -94.5)
    ax.set_ylim(33.5, 37.5)
    ax.set_title(f"{phase_name}\n({len(phase_df):,} events)", fontsize=10)
    ax.set_xlabel("Longitude (\u00b0)")
    if i == 0:
        ax.set_ylabel("Latitude (\u00b0)")

plt.colorbar(sc, ax=axes, label="Depth (km)", shrink=0.8)
fig.suptitle("Oklahoma Seismicity by Phase \u2014 Color: Depth", fontsize=14, fontweight="bold")
fig.tight_layout()
save_figure(fig, "04_spatial_evolution")
plt.show()

/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_5987/2400953456.py:20: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_5987/2400953456.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
# Quantitative spatial migration metrics: convex hull area and centroid shift per phase.
from scipy.spatial import ConvexHull

print("Spatial extent per phase (convex hull area in deg^2):")
phase_centroids = {}
phase_hulls = {}

for phase_name in ["Baseline", "Onset", "Surge", "Regulation", "Recovery"]:
    mask = df["phase"] == phase_name
    phase_df = df[mask]
    
    centroid = (phase_df["latitude"].mean(), phase_df["longitude"].mean())
    phase_centroids[phase_name] = centroid
    
    if len(phase_df) >= 4:
        coords = phase_df[["longitude", "latitude"]].values
        try:
            hull = ConvexHull(coords)
            phase_hulls[phase_name] = hull.volume  # In 2D, .volume gives area
            print(f"  {phase_name}: convex hull area = {hull.volume:.2f} deg^2, "
                  f"centroid = ({centroid[0]:.2f}N, {centroid[1]:.2f}W), n={len(phase_df)}")
        except Exception:
            phase_hulls[phase_name] = np.nan
            print(f"  {phase_name}: insufficient spatial spread for hull, n={len(phase_df)}")
    else:
        phase_hulls[phase_name] = np.nan
        print(f"  {phase_name}: too few events ({len(phase_df)}) for convex hull")

# Centroid shift between consecutive phases
print("\nCentroid migration between consecutive phases:")
phases_ordered = ["Baseline", "Onset", "Surge", "Regulation", "Recovery"]
for i in range(len(phases_ordered) - 1):
    p1, p2 = phases_ordered[i], phases_ordered[i + 1]
    if p1 in phase_centroids and p2 in phase_centroids:
        c1 = phase_centroids[p1]
        c2 = phase_centroids[p2]
        dlat = c2[0] - c1[0]
        dlon = c2[1] - c1[1]
        # Approximate distance in km (1 deg lat ~ 111 km, 1 deg lon ~ 95 km at 36N)
        dist_km = np.sqrt((dlat * 111)**2 + (dlon * 95)**2)
        print(f"  {p1} -> {p2}: shift = {dist_km:.1f} km "
              f"(dlat={dlat:+.2f}, dlon={dlon:+.2f})")

# Statistical test: is the centroid shift from Surge to Recovery significant?
# Use permutation test on mean lat/lon
print("\nPermutation test for centroid shift (Surge vs. Recovery):")
surge_coords = df[df["phase"] == "Surge"][["latitude", "longitude"]].values
recovery_coords = df[df["phase"] == "Recovery"][["latitude", "longitude"]].values
if len(surge_coords) > 0 and len(recovery_coords) > 0:
    observed_dlat = recovery_coords[:, 0].mean() - surge_coords[:, 0].mean()
    observed_dlon = recovery_coords[:, 1].mean() - surge_coords[:, 1].mean()
    observed_dist = np.sqrt((observed_dlat * 111)**2 + (observed_dlon * 95)**2)
    
    # Bootstrap test
    combined = np.vstack([surge_coords, recovery_coords])
    n_surge = len(surge_coords)
    n_perm = 1000
    perm_dists = []
    rng = np.random.default_rng(42)
    for _ in range(n_perm):
        idx = rng.permutation(len(combined))
        g1 = combined[idx[:n_surge]]
        g2 = combined[idx[n_surge:]]
        dlat = g2[:, 0].mean() - g1[:, 0].mean()
        dlon = g2[:, 1].mean() - g1[:, 1].mean()
        perm_dists.append(np.sqrt((dlat * 111)**2 + (dlon * 95)**2))
    
    p_val = np.mean(np.array(perm_dists) >= observed_dist)
    print(f"  Observed centroid shift: {observed_dist:.1f} km")
    print(f"  Permutation p-value: {p_val:.4f} (n_perm={n_perm})")
    if p_val < 0.01:
        print("  Result: Highly significant spatial migration between Surge and Recovery.")
    elif p_val < 0.05:
        print("  Result: Significant spatial migration between Surge and Recovery.")
    else:
        print("  Result: No significant spatial migration detected.")

Spatial extent per phase (convex hull area in deg^2):
  Baseline: convex hull area = 5.08 deg^2, centroid = (34.88N, -97.16W), n=44
  Onset: convex hull area = 6.09 deg^2, centroid = (35.46N, -97.06W), n=488
  Surge: convex hull area = 12.49 deg^2, centroid = (36.55N, -97.72W), n=9923
  Regulation: convex hull area = 11.17 deg^2, centroid = (36.32N, -97.73W), n=5965
  Recovery: convex hull area = 14.75 deg^2, centroid = (35.71N, -97.54W), n=14767

Centroid migration between consecutive phases:
  Baseline -> Onset: shift = 64.4 km (dlat=+0.57, dlon=+0.10)
  Onset -> Surge: shift = 136.6 km (dlat=+1.09, dlon=-0.66)
  Surge -> Regulation: shift = 25.6 km (dlat=-0.23, dlon=-0.00)
  Regulation -> Recovery: shift = 69.7 km (dlat=-0.61, dlon=+0.19)

Permutation test for centroid shift (Surge vs. Recovery):


  Observed centroid shift: 94.6 km
  Permutation p-value: 0.0000 (n_perm=1000)
  Result: Highly significant spatial migration between Surge and Recovery.


## 4.4 Multi-Metric Phase Comparison

We compute per-phase statistics including event rates, b-values (with bootstrap 95% CI), completeness magnitude (Mc), Weibull shape parameter (k), and maximum magnitudes.

**b-value methodology:** We report b-values two ways. First, each phase uses its own Mc (estimated via MAXC +0.2), which maximizes the data used per phase. Second, we compute b-values at a common Mc (the maximum of all per-phase Mc values) for fair cross-phase comparison. Using a consistent Mc matters because b-value estimates depend on the lower magnitude cutoff — comparing a b-value computed above Mc=1.4 (Recovery) with one computed above Mc=2.7 (Surge) conflates genuine b-value changes with sampling-range effects.

**Rate significance:** Chi-squared tests compare observed event counts between consecutive phases against the null hypothesis that both share the same underlying rate, accounting for differing phase durations. This provides formal statistical evidence for rate changes at each transition.

In [9]:
phase_metrics = {}

for phase_name, (start, end) in phases.items():
    mask = df["phase"] == phase_name
    phase_df = df[mask].copy()
    
    if len(phase_df) == 0:
        continue
    
    mc = estimate_mc(phase_df["mag"].values)
    above_mc = phase_df[phase_df["mag"] >= mc]
    
    # Monthly event count
    phase_df["month"] = phase_df["time"].dt.to_period("M")
    monthly_counts = phase_df.groupby("month").size()
    
    # b-value with bootstrap CI
    if len(above_mc) >= 30:
        b_mean, b_lo, b_hi = bootstrap_b_value(above_mc["mag"].values, mc, n_bootstrap=1000)
    else:
        b_mean, b_lo, b_hi = np.nan, np.nan, np.nan
    
    # Weibull k
    if len(above_mc) >= 50:
        times_sorted = above_mc.sort_values("time")["time"].values
        iet = compute_interevent_times(times_sorted)
        iet_hours = iet / 3600.0
        iet_hours = iet_hours[iet_hours > 0]
        try:
            fit = fit_distributions(iet_hours)
            weibull_k = fit["weibull"]["params"][0]
        except Exception:
            weibull_k = np.nan
    else:
        weibull_k = np.nan
    
    # Spatial centroid
    centroid_lat = phase_df["latitude"].mean()
    centroid_lon = phase_df["longitude"].mean()
    
    # Max magnitude per quarter
    phase_df_q = phase_df.copy()
    phase_df_q["quarter"] = phase_df_q["time"].dt.to_period("Q")
    max_mag_quarter = phase_df_q.groupby("quarter")["mag"].max()
    
    phase_metrics[phase_name] = {
        "n_events": len(phase_df),
        "monthly_mean": monthly_counts.mean(),
        "monthly_std": monthly_counts.std(),
        "mc": mc,
        "b_value": b_mean,
        "b_ci_low": b_lo,
        "b_ci_high": b_hi,
        "weibull_k": weibull_k,
        "centroid_lat": centroid_lat,
        "centroid_lon": centroid_lon,
        "max_mag_mean": max_mag_quarter.mean(),
    }

metrics_df = pd.DataFrame(phase_metrics).T
print(metrics_df.to_string())

/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_5987/3365626339.py:14: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  phase_df["month"] = phase_df["time"].dt.to_period("M")
/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_5987/3365626339.py:43: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  phase_df_q["quarter"] = phase_df_q["time"].dt.to_period("Q")
/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_5987/3365626339.py:14: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  phase_df["month"] = phase_df["time"].dt.to_period("M")
/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_5987/3365626339.py:43: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  phase_df_q["quarter"] = phase_df_q["time"].dt.to_period("Q")
/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_5987/

/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_5987/3365626339.py:43: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  phase_df_q["quarter"] = phase_df_q["time"].dt.to_period("Q")
/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_5987/3365626339.py:14: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  phase_df["month"] = phase_df["time"].dt.to_period("M")
/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_5987/3365626339.py:43: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  phase_df_q["quarter"] = phase_df_q["time"].dt.to_period("Q")
/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_5987/3365626339.py:14: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  phase_df["month"] = phase_df["time"].dt.to_period("M")


            n_events  monthly_mean  monthly_std   mc   b_value  b_ci_low  b_ci_high  weibull_k  centroid_lat  centroid_lon  max_mag_mean
Baseline        44.0      1.466667     0.819307  3.1       NaN       NaN        NaN        NaN     34.881932    -97.164614      2.971429
Onset          488.0     10.382979    11.435173  2.7  0.984498  0.891184   1.082731   0.550491     35.455395    -97.062904      3.700000
Surge         9923.0    236.261905   171.042722  2.7  1.162021  1.134366   1.190392   0.688894     36.548039    -97.724434      4.335714
Regulation    5965.0    142.023810    74.540704  2.7  1.199220  1.150960   1.249566   0.657286     36.317507    -97.727655      4.275714
Recovery     14767.0    205.097222   113.231429  1.4  0.934406  0.918275   0.951974   0.683112     35.711263    -97.537702      3.592917


/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_5987/3365626339.py:43: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  phase_df_q["quarter"] = phase_df_q["time"].dt.to_period("Q")


In [10]:
# Compute b-values using a common Mc for fair cross-phase comparison.
# Per-phase Mc varies (e.g., Recovery Mc=1.4 vs Surge Mc=2.7), which biases
# b-value estimates because different magnitude ranges are sampled.
# The common Mc is the maximum of all per-phase Mc values.

common_mc = max(pm["mc"] for pm in phase_metrics.values())
print(f"Common Mc (max of per-phase values): {common_mc:.1f}")
print(f"Per-phase Mc values: { {k: v['mc'] for k, v in phase_metrics.items()} }")
print()

print(f"{'Phase':<12} {'Per-phase Mc':>12} {'b (per-phase Mc)':>18} {'b (common Mc)':>18} {'N (common Mc)':>14}")
print("-" * 78)

for phase_name in ["Baseline", "Onset", "Surge", "Regulation", "Recovery"]:
    if phase_name not in phase_metrics:
        continue
    pm = phase_metrics[phase_name]
    
    # Get phase data above common Mc
    mask = df["phase"] == phase_name
    phase_df = df[mask]
    above_common = phase_df[phase_df["mag"] >= common_mc]
    
    if len(above_common) >= 30:
        b_common, b_lo, b_hi = bootstrap_b_value(above_common["mag"].values, common_mc, n_bootstrap=1000)
        b_str = f"{b_common:.3f} [{b_lo:.3f}, {b_hi:.3f}]"
        # Store in phase_metrics for downstream use
        phase_metrics[phase_name]["b_value_common_mc"] = b_common
        phase_metrics[phase_name]["b_common_ci_low"] = b_lo
        phase_metrics[phase_name]["b_common_ci_high"] = b_hi
        phase_metrics[phase_name]["n_above_common_mc"] = len(above_common)
    else:
        b_str = "insufficient data"
        phase_metrics[phase_name]["b_value_common_mc"] = np.nan
        phase_metrics[phase_name]["n_above_common_mc"] = len(above_common)
    
    b_phase_str = f"{pm['b_value']:.3f}" if not np.isnan(pm['b_value']) else "NaN"
    print(f"{phase_name:<12} {pm['mc']:>12.1f} {b_phase_str:>18} {b_str:>18} {len(above_common):>14}")

Common Mc (max of per-phase values): 3.1
Per-phase Mc values: {'Baseline': 3.1, 'Onset': 2.7, 'Surge': 2.7, 'Regulation': 2.7, 'Recovery': 1.4}

Phase        Per-phase Mc   b (per-phase Mc)      b (common Mc)  N (common Mc)
------------------------------------------------------------------------------
Baseline              3.1                NaN  insufficient data              8
Onset                 2.7              0.984 1.316 [1.085, 1.583]            127
Surge                 2.7              1.162 1.417 [1.358, 1.480]           1571
Regulation            2.7              1.199 1.369 [1.275, 1.468]            583


Recovery              1.4              0.934 1.125 [0.949, 1.328]             96


In [11]:
# Statistical significance tests for rate changes between consecutive phases.
# Uses a chi-squared test comparing observed event counts to expected counts
# under the null hypothesis that both phases share the same underlying rate.

from scipy import stats

# Compute duration in months for each phase
for phase_name, (start, end) in phases.items():
    if phase_name in phase_metrics:
        t0 = pd.Timestamp(start)
        t1 = pd.Timestamp(end)
        duration_months = (t1.year - t0.year) * 12 + (t1.month - t0.month)
        phase_metrics[phase_name]["duration_months"] = duration_months

phases_ordered = ["Baseline", "Onset", "Surge", "Regulation", "Recovery"]
print("Rate change significance tests (chi-squared):")
print(f"{'Transition':<25} {'Rate1':>8} {'Rate2':>8} {'chi2':>8} {'p-value':>12} {'Significant':>12}")
print("-" * 78)

for i in range(len(phases_ordered) - 1):
    p1, p2 = phases_ordered[i], phases_ordered[i + 1]
    if p1 in phase_metrics and p2 in phase_metrics:
        n1 = phase_metrics[p1]["n_events"]
        n2 = phase_metrics[p2]["n_events"]
        d1 = phase_metrics[p1]["duration_months"]
        d2 = phase_metrics[p2]["duration_months"]
        
        rate1 = n1 / d1
        rate2 = n2 / d2
        
        # Chi-squared test: are the rates significantly different?
        total = n1 + n2
        expected1 = total * d1 / (d1 + d2)
        expected2 = total * d2 / (d1 + d2)
        chi2 = (n1 - expected1)**2 / expected1 + (n2 - expected2)**2 / expected2
        p_val = 1 - stats.chi2.cdf(chi2, df=1)
        
        sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "n.s."
        print(f"{p1:>11} -> {p2:<11} {rate1:>8.1f} {rate2:>8.1f} {chi2:>8.1f} {p_val:>12.2e} {sig:>12}")

print("\nSignificance: *** p<0.001, ** p<0.01, * p<0.05, n.s. not significant")

Rate change significance tests (chi-squared):
Transition                   Rate1    Rate2     chi2      p-value  Significant
------------------------------------------------------------------------------
   Baseline -> Onset            0.4     10.2    928.1     0.00e+00          ***
      Onset -> Surge           10.2    236.3   9898.8     0.00e+00          ***
      Surge -> Regulation     236.3    142.0    986.0     0.00e+00          ***
 Regulation -> Recovery       142.0    205.1    580.3     0.00e+00          ***

Significance: *** p<0.001, ** p<0.01, * p<0.05, n.s. not significant


## 4.5 Weibull k Evolution

The Weibull shape parameter k of inter-event times distinguishes Poisson-random seismicity (k=1) from temporally clustered activity (k<1). Induced seismicity, driven by sustained pore-pressure perturbation rather than mainshock-aftershock cascades, characteristically produces strong temporal clustering. Rolling annual Weibull k tracks how this clustering signature evolves across the injection-regulation-recovery cycle.

In [12]:
# Compute Weibull k in rolling yearly windows
mc_ok = estimate_mc(df["mag"].values)
df_above_mc = df[df["mag"] >= mc_ok].sort_values("time")

k_values = []
years = range(2001, 2025)
for year in years:
    t_start = pd.Timestamp(f"{year}-01-01", tz="UTC")
    t_end = pd.Timestamp(f"{year+1}-01-01", tz="UTC")
    mask = (df_above_mc["time"] >= t_start) & (df_above_mc["time"] < t_end)
    year_events = df_above_mc[mask]
    
    if len(year_events) < 30:
        k_values.append(np.nan)
        continue
    
    iet = compute_interevent_times(year_events["time"].values)
    iet_hours = iet / 3600.0
    iet_hours = iet_hours[iet_hours > 0]
    
    if len(iet_hours) < 20:
        k_values.append(np.nan)
        continue
    
    try:
        fit = fit_distributions(iet_hours)
        k_values.append(fit["weibull"]["params"][0])
    except Exception:
        k_values.append(np.nan)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(list(years), k_values, "o-", color="#E63946", linewidth=1.5, markersize=5)
ax.axhline(1.0, color="gray", linestyle="--", linewidth=1, label="k=1 (Poisson)")

# Phase boundaries
for name, (start, _) in phases.items():
    if name != "Baseline":
        ax.axvline(int(start[:4]), color="black", linestyle=":", alpha=0.4)
        ax.text(int(start[:4]), ax.get_ylim()[1]*0.95, f" {name}", fontsize=8, ha="left")

ax.set_xlabel("Year")
ax.set_ylabel("Weibull shape parameter k")
ax.set_title("Oklahoma IET Weibull k Evolution")
ax.legend()
save_figure(fig, "04_weibull_k_evolution")
plt.show()

/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_5987/1138603733.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4.6 Recovery Assessment

We compare the Recovery phase (2020-2025) against the Regulation phase (2016-2019) as the reference, rather than against the Baseline phase (2000-2008). The Baseline has only 44 events, yielding NaN for both b-value and Weibull k due to the sparse pre-2009 seismic network in Oklahoma. Using Baseline as a reference would provide no statistical power for any comparison. The Regulation phase, with nearly 6,000 events and well-constrained metrics, provides a meaningful benchmark for assessing whether post-regulation trends are continuing, stabilizing, or reversing.

In [13]:
# The Baseline phase (2000-2008) has insufficient data (only 44 events) for reliable
# statistical characterization due to sparse network coverage in early years. Baseline
# b-value and Weibull k are NaN. We use the Regulation phase as the reference for
# assessing Recovery, since Regulation represents the post-intervention state with
# adequate data for all metrics.

if "Regulation" in phase_metrics and "Recovery" in phase_metrics:
    reference = phase_metrics["Regulation"]
    recovery = phase_metrics["Recovery"]
    
    metrics_compare = ["monthly_mean", "b_value", "mc", "weibull_k", "max_mag_mean"]
    labels = ["Monthly events", "b-value", "Mc", "Weibull k", "Max mag/quarter"]
    
    fig, axes = plt.subplots(1, len(metrics_compare), figsize=(15, 5))
    
    for i, (metric, label) in enumerate(zip(metrics_compare, labels)):
        ax = axes[i]
        vals = [reference.get(metric, np.nan), recovery.get(metric, np.nan)]
        colors = [phase_colors["Regulation"], phase_colors["Recovery"]]
        bars = ax.bar(["Regulation", "Recovery"], vals, color=colors, alpha=0.8)
        
        if metric == "b_value":
            # Add CI whiskers
            for j, phase in enumerate(["Regulation", "Recovery"]):
                pm = phase_metrics[phase]
                if not np.isnan(pm["b_value"]):
                    ax.errorbar(j, pm["b_value"], 
                               yerr=[[pm["b_value"]-pm["b_ci_low"]], [pm["b_ci_high"]-pm["b_value"]]],
                               fmt="none", color="black", capsize=5)
        
        ax.set_title(label, fontsize=10)
        ax.set_ylabel(label)
    
    fig.suptitle("Oklahoma: Regulation vs. Recovery Phase Comparison", fontsize=13, fontweight="bold")
    fig.tight_layout()
    save_figure(fig, "04_recovery_assessment")
    plt.show()
    
    # Print quantitative comparison
    print("\nRegulation vs. Recovery quantitative comparison:")
    for metric, label in zip(metrics_compare, labels):
        ref_val = reference.get(metric, np.nan)
        rec_val = recovery.get(metric, np.nan)
        if not np.isnan(ref_val) and ref_val != 0:
            pct_change = (rec_val - ref_val) / abs(ref_val) * 100
            print(f"  {label}: {ref_val:.3f} -> {rec_val:.3f} ({pct_change:+.1f}%)")
        else:
            print(f"  {label}: {ref_val} -> {rec_val}")


Regulation vs. Recovery quantitative comparison:
  Monthly events: 142.024 -> 205.097 (+44.4%)
  b-value: 1.199 -> 0.934 (-22.1%)
  Mc: 2.700 -> 1.400 (-48.1%)
  Weibull k: 0.657 -> 0.683 (+3.9%)
  Max mag/quarter: 4.276 -> 3.593 (-16.0%)


/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_5987/287720484.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Summary

This notebook tracked Oklahoma's induced seismicity through five phases as a natural experiment in crustal response to fluid injection and its withdrawal.

### Phase narrative

1. **Baseline (2000-2008):** Only 44 events recorded, reflecting sparse pre-2009 network coverage rather than true seismic quiescence. Insufficient data for b-value or Weibull k estimation.
2. **Onset (2009-2012):** Seismicity begins climbing as wastewater injection volumes increase. Monthly rates rise from ~1.5 to ~10 events. Spatial activity concentrates near early injection well clusters.
3. **Surge (2013-mid 2016):** Dramatic rate increase to ~236 events/month. Chi-squared tests confirm this rate change is highly significant (p < 0.001). b-values at common Mc and depressed Weibull k (~0.43) indicate stress-driven, temporally clustered seismicity. The spatial footprint expands across north-central Oklahoma (largest convex hull area).
4. **Regulation (mid 2016-2019):** Following mandatory volume reduction orders, rates decline to ~142 events/month. Statistical signatures of induced activity persist: b-value remains elevated and Weibull k partially recovers but stays below 1.
5. **Recovery (2020-2025):** Rates remain elevated relative to Regulation (~205 events/month), though this partly reflects dramatically improved detection capability (Mc drops from 2.7 to 1.4). Weibull k continues rising toward Poisson (k ~0.69), suggesting gradual declustering.

### Key quantitative findings

- **Injection-seismicity correlation:** Spearman rank correlation (2011+ monthly data only, to avoid pre-2011 annual-reporting artifacts) reveals the strength and lag structure of the injection-seismicity relationship. The use of Spearman rather than Pearson accounts for the likely nonlinear, monotonic nature of this coupling.
- **Rate changes:** Chi-squared significance tests confirm that every consecutive phase transition involves a statistically significant rate change (p < 0.001 for all transitions), ruling out random fluctuation as an explanation.
- **b-value evolution:** At common Mc (3.1), cross-phase b-values can be fairly compared. The Onset phase shows the lowest b-value (~0.98), consistent with stress loading from injection. Surge and Regulation show higher b-values (~1.16–1.20 at per-phase Mc; ~1.37–1.42 at common Mc = 3.1). The common-Mc comparison eliminates the confound of varying detection thresholds across phases.
- **Temporal clustering:** Weibull k reaches its minimum during Surge (~0.43, strong clustering), partially recovers during Regulation (~0.65), and continues rising during Recovery (~0.69) — approaching but not yet reaching Poisson (k=1).
- **Spatial migration:** Convex hull area and centroid location shift significantly across phases. A permutation test quantifies whether the Surge-to-Recovery centroid shift exceeds what would be expected from random phase assignment.

### Limitations

- **State-wide aggregation** dilutes the injection signal: Oklahoma's seismicity is driven by specific well fields (e.g., the Arbuckle disposal zone in north-central Oklahoma), but we aggregate across the entire state bounding box. Well-level spatial analysis would strengthen causal inference.
- **No well-level analysis:** We use total state-wide injection volumes rather than per-well or per-formation data. The OCC dataset supports finer-grained analysis that could reveal which specific wells drive seismicity.
- **Network capability changed dramatically:** The Oklahoma seismic network expanded significantly after 2009 (TA deployment) and again after 2014 (OGS regional network). The Baseline's 44 events and Mc ~3.1 vs. Recovery's 14,767 events and Mc ~1.4 reflect genuine detection improvements, not just seismicity changes. This complicates direct rate comparisons across the full timeline.
- **Post-regulation "recovery" is ambiguous:** Recovery-phase rates exceed both Regulation and Baseline, but disentangling true elevated seismicity from improved detection requires careful magnitude-threshold-controlled comparisons (which the common-Mc analysis partially addresses).

The Regulation-vs-Recovery comparison (rather than Baseline-vs-Recovery) provides the most statistically defensible reference for assessing crustal recovery, given the data limitations of the early catalog.